In [1]:
import pandas as pd
import csv

# --- INPUT PATHS ---
bbg_xlsx = r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\exchanges-code-bloomberg.xlsx"
iso_csv  = r"D:\LinhDao\Programming\SUPERFUNdProject\iso_country_codes_complete.csv"

# --- OUTPUT PATH ---
out_csv  = r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\bloomberg-exchange-codes-full.csv"

# 1) Load Bloomberg Excel (sheet "MIC") and normalize headers
#    IMPORTANT: keep_default_na=False and na_values=[] so literal 'NA' stays 'NA'
bbg = pd.read_excel(
    bbg_xlsx,
    sheet_name="MIC",
    dtype=str,
    keep_default_na=False,
    na_values=[]
)
bbg.columns = (bbg.columns
               .str.replace("\u00A0"," ", regex=False)
               .str.strip()
               .str.upper())

# 2) Pick out the three Bloomberg columns we want
needed_cols = ["EQUITY EXCH CODE", "ISO COUNTRY", "COMPOSITE CODE"]
missing_cols = [c for c in needed_cols if c not in bbg.columns]
if missing_cols:
    raise KeyError(f"Missing column(s) in Excel: {missing_cols}")

bbg = bbg[needed_cols].copy().rename(columns={
    "EQUITY EXCH CODE": "BBG_Code",
    "ISO COUNTRY": "ISO_Alpha2",
    "COMPOSITE CODE": "Composite_Code"
})

# Clean values (remove NBSPs, strip, uppercase)
for c in ["BBG_Code", "ISO_Alpha2", "Composite_Code"]:
    bbg[c] = (bbg[c].astype(str)
                      .str.replace("\u00A0"," ", regex=False)
                      .str.strip()
                      .str.upper())

# 3) Load ISO lookup and normalize
iso = pd.read_csv(
    iso_csv,
    dtype=str,
    keep_default_na=False,
    na_values=[],
    encoding="utf-8-sig"
)
iso.columns = iso.columns.str.strip()
iso = iso[["Alpha-2", "Country (Friendly)"]].rename(columns={"Alpha-2": "ISO_Alpha2"})
iso["ISO_Alpha2"] = iso["ISO_Alpha2"].str.strip().str.upper()

# 4) Merge to attach friendly country names
merged = bbg.merge(iso, on="ISO_Alpha2", how="left")

# Apply overrides for confusing ISO names
overrides = {
    "Taiwan, Province of China": "Taiwan",
    "Côte d’Ivoire": "Ivory Coast"
}
merged["Country (Friendly)"] = merged["Country (Friendly)"].replace(overrides)

# --- Sanity: literal 'NA' counts BEFORE saving ---
print("Literal 'NA' counts by column (pre-save):")
for col in merged.columns:
    print(f"  {col}: {(merged[col] == 'NA').sum()}")

# 5) Save to CSV
#    - fill true nulls with empty string (does not touch literal 'NA' strings)
#    - quote all fields so Excel treats 'NA' as text, not missing
merged = merged.fillna("")
merged.to_csv(out_csv, index=False, encoding="utf-8-sig", quoting=csv.QUOTE_ALL)

# 6) Summary
print("✅ Saved:", out_csv)
print(f"Rows: {len(merged)} | Missing friendly names: {merged['Country (Friendly)'].eq('').sum()}")
print("Count of BBG_Code == 'NA':", (merged["BBG_Code"] == "NA").sum())
print("Count of Composite_Code == 'NA':", (merged["Composite_Code"] == "NA").sum())
print(merged.head(12).to_string(index=False))


Literal 'NA' counts by column (pre-save):
  BBG_Code: 1
  ISO_Alpha2: 1
  Composite_Code: 1
  Country (Friendly): 0
✅ Saved: D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\bloomberg-exchange-codes-full.csv
Rows: 426 | Missing friendly names: 0
Count of BBG_Code == 'NA': 1
Count of Composite_Code == 'NA': 1
BBG_Code ISO_Alpha2 Composite_Code Country (Friendly)
      AJ         ZA             SJ       South Africa
      PF         AU             AU          Australia
      UP         US             US      United States
      AQ         AU             AU          Australia
      QE         FR             QE             France
      QX         GB             QX     United Kingdom
      EB         GB             EB     United Kingdom
      UF         US             US      United States
      VY         US             US      United States
      RB         BY             RB            Belarus
      E2         NL             E2        Netherlands
      MU         MX             